# Telco Customer Churn
## Análisis Exploratorio de Datos

Este notebook documenta el proceso de comprensión, evaluación de calidad, preparación y análisis exploratorio del conjunto de datos Telco Customer Churn

## Contexto y alcance del proyecto

El proyecto analiza el abandono de clientes de una empresa de telecomunicaciones mediante el conjunto de datos Telco Customer Churn.

El objetivo es analizar los **7.043 clientes** del conjunto de datos, cuantificar la magnitud de `Churn`, evaluar la calidad de la información e identificar patrones y segmentos con diferencias relevantes en la tasa de abandono.

El principal KPI utilizado como referencia es la **tasa global de abandono**, correspondiente a la proporción de clientes con `Churn = Yes`.

El alcance de esta evaluación corresponde a comprensión del problema, preparación de datos, análisis exploratorio, privacidad, ética y posibles sesgos. **No se realiza entrenamiento de modelos en esta etapa.**

## 1. Importación de librerías

In [9]:
#Librerías necesarias para manipulación, análisis numérico y visualización de datos
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

## 2. Carga y comprensión inicial del conjunto de datos

In [10]:
#Carga del dataset original
# Descarga el archivo desde GitHub al entorno de ejecución
!wget -O Telco_Customer_Churn_Dataset.csv "https://raw.githubusercontent.com/eduardodrm/machine_learning_ev1/main/data/Telco_Customer_Churn_Dataset.csv"

data = pd.read_csv("/content/Telco_Customer_Churn_Dataset.csv")

--2026-09-06 12:56:19--  https://raw.githubusercontent.com/eduardodrm/machine_learning_ev1/main/data/Telco_Customer_Churn_Dataset.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 977501 (955K) [text/plain]
Saving to: ‘Telco_Customer_Churn_Dataset.csv’

Telco_Customer_Chur 100%[===================>] 954.59K  --.-KB/s    in 0.05s   

2026-09-06 12:56:19 (20.2 MB/s) - ‘Telco_Customer_Churn_Dataset.csv’ saved [977501/977501]



In [11]:
#Visualización de las primeras filas del dataset
data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [12]:
#Dimensiones del dataset
data.shape

(7043, 21)

In [13]:
#Información estructural del dataset
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


### 2.1 Comprensión inicial de la estructura

El conjunto de datos contiene **7.043 observaciones y 21 variables**, correspondientes a información demográfica, servicios contratados, características de la relación comercial y cargos asociados a los clientes.

La revisión de los tipos de datos muestra variables almacenadas como `object`, `int64` y `float64`. Se observa particularmente que `TotalCharges`, pese a representar un monto acumulado, se encuentra almacenada como `object`, por lo que será necesario investigar su contenido antes de realizar cualquier transformación.

In [14]:
#Nombres de las variables disponibles en el dataset
data.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

## 3. Auditoría de calidad de los datos

### 3.1 Completitud y valores ausentes

In [15]:
#Revisión de valores ausentes reconocidos por Pandas
data.isna().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [16]:
#Verificación del tipo almacenado de TotalCharges
data["TotalCharges"].dtype

dtype('O')

In [17]:
#Cantidad de registros con espacios en blanco en TotalCharges
(data["TotalCharges"].str.strip() == "").sum()

np.int64(11)

In [18]:
#Inspección de los registros con TotalCharges en blanco
data.loc[
    data["TotalCharges"].str.strip() == "",
    ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]
]

,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


La revisión inicial mediante `isna()` no identificó valores ausentes reconocidos por Pandas. Sin embargo, al investigar la variable `TotalCharges`, almacenada como `object` pese a representar un monto, se identificaron **11 registros con espacios en blanco**.

Estos valores no habían sido reconocidos inicialmente como nulos. Además, los 11 registros afectados presentan `tenure = 0`, por lo que esta condición deberá analizarse antes de definir el tratamiento correspondiente.

Este hallazgo evidencia que la evaluación de completitud no debe limitarse únicamente a los valores `NaN` detectados automáticamente.

### 3.2 Unicidad y registros duplicados

In [19]:
#Cantidad de registros completamente duplicados
data.duplicated().sum()

np.int64(0)

In [20]:
#Comparación entre número de registros e identificadores únicos
print("Cantidad de registros:", len(data))
print("CustomerID únicos:", data["customerID"].nunique())

Cantidad de registros: 7043
CustomerID únicos: 7043


La revisión de unicidad no identificó registros completamente duplicados.

Además, `customerID` presenta 7.043 valores únicos para las 7.043 observaciones del conjunto de datos, sin identificadores repetidos. Esto es consistente con su función como identificador individual de cada cliente.

Por lo tanto, no se requiere realizar eliminación de registros por duplicidad.

### 3.3 Validación de categorías, rangos y consistencia

In [21]:
#Revisión de los valores presentes en SeniorCitizen
data["SeniorCitizen"].value_counts(dropna=False)

,count
SeniorCitizen,
0,5901
1,1142


In [22]:
#Revisión de las categorías presentes en Churn
data["Churn"].value_counts(dropna=False)

,count
Churn,
No,5174
Yes,1869


In [23]:
#Revisión de las categorías presentes en Contract
data["Contract"].value_counts(dropna=False)

,count
Contract,
Month-to-month,3875
Two year,1695
One year,1473


### 3.4 Tratamiento de la variable `TotalCharges`

Durante la evaluación de calidad se identificaron 11 registros representados mediante espacios en blanco en `TotalCharges`. Debido a esta situación, la variable había sido almacenada como `object` pese a representar un monto acumulado.

Se procede a convertir la variable a formato numérico, transformando temporalmente los valores no convertibles en valores ausentes para poder analizarlos y tratarlos de manera explícita.

In [24]:
#Conversión de TotalCharges a tipo numérico
data["TotalCharges"] = pd.to_numeric(
    data["TotalCharges"],
    errors="coerce"
)

In [25]:
#Verificación del nuevo tipo de dato y valores ausentes generados
print("Tipo de dato:", data["TotalCharges"].dtype)
print("Valores ausentes:", data["TotalCharges"].isna().sum())

Tipo de dato: float64
Valores ausentes: 11


In [26]:
#Asignación de 0 exclusivamente a los casos ausentes con tenure igual a 0
data.loc[
    data["TotalCharges"].isna() & (data["tenure"] == 0),
    "TotalCharges"
] = 0

In [27]:
#Validación posterior al tratamiento
print("Valores ausentes en TotalCharges:", data["TotalCharges"].isna().sum())
print("Tipo de dato de TotalCharges:", data["TotalCharges"].dtype)

Valores ausentes en TotalCharges: 0
Tipo de dato de TotalCharges: float64


Los 11 valores no convertibles de `TotalCharges` correspondían exclusivamente a clientes con `tenure = 0`. Considerando que `tenure` representa los meses de permanencia y `TotalCharges` los cargos acumulados, se asignó un valor de `0` únicamente a estos registros.

La imputación se realizó de manera condicionada, evitando reemplazar indiscriminadamente cualquier valor ausente que pudiera presentar una causa diferente.

Finalmente, se verificó que `TotalCharges` quedara almacenada como variable numérica (`float64`) y sin valores ausentes.

### 3.5 Validación de rangos numéricos

In [28]:
#Revisión de valores mínimos y máximos de variables numéricas relevantes
data[["tenure", "MonthlyCharges", "TotalCharges"]].agg(["min", "max"])

,tenure,MonthlyCharges,TotalCharges
min,0,18.25,0.0
max,72,118.75,8684.8


In [29]:
#Verificación de valores negativos en variables numéricas relevantes
(data[["tenure", "MonthlyCharges", "TotalCharges"]] < 0).sum()

,0
tenure,0
MonthlyCharges,0
TotalCharges,0


Las variables numéricas `tenure`, `MonthlyCharges` y `TotalCharges` no presentan valores negativos.

Los rangos observados son de 0 a 72 meses para `tenure`, de 18,25 a 118,75 para `MonthlyCharges` y de 0 a 8.684,80 para `TotalCharges`.

No se identificaron valores evidentemente inválidos a partir de esta revisión de rangos.

### 3.6 Validación de consistencia lógica entre variables

In [30]:
#Consistencia entre PhoneService y MultipleLines
data.loc[
    data["PhoneService"] == "No",
    "MultipleLines"
].value_counts(dropna=False)

,count
MultipleLines,
No phone service,682


In [31]:
# Consistencia entre InternetService y servicios asociados
data.loc[
    data["InternetService"] == "No",
    [
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies"
    ]
].drop_duplicates()

,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies
11,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service


Se revisó la consistencia entre variables relacionadas con la contratación de servicios.

Los clientes con `PhoneService = No` presentan de manera consistente la categoría `No phone service` en `MultipleLines`.

De forma similar, los clientes con `InternetService = No` presentan la categoría `No internet service` en las variables asociadas a servicios de Internet. Por lo tanto, estas categorías representan situaciones válidas de no aplicabilidad y no deben interpretarse como valores ausentes.

## 4. Análisis estadístico descriptivo

### 4.1 Variables numéricas

In [32]:
#Resumen estadístico de las variables numéricas
data.describe()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692,2279.734304
std,0.368612,24.559481,30.090047,2266.794470
min,0.000000,0.000000,18.250000,0.000000
25%,0.000000,9.000000,35.500000,398.550000
50%,0.000000,29.000000,70.350000,1394.550000
75%,0.000000,55.000000,89.850000,3786.600000
max,1.000000,72.000000,118.750000,8684.800000


El resumen estadístico muestra que los clientes presentan una permanencia promedio (`tenure`) de aproximadamente **32,37 meses**, con una mediana de **29 meses** y una desviación estándar de **24,56 meses**. El 50 % central de los clientes presenta una permanencia comprendida entre **9 y 55 meses**.

El cargo mensual (`MonthlyCharges`) presenta una media aproximada de **64,76** y una mediana de **70,35**, con valores comprendidos entre **18,25 y 118,75**.

Por su parte, `TotalCharges` presenta una media aproximada de **2.279,73**, una mediana de **1.394,55** y una desviación estándar de **2.266,79**, lo que evidencia una elevada variabilidad en los cargos totales registrados.

La variable `SeniorCitizen`, aunque está almacenada como `int64`, corresponde conceptualmente a una variable binaria codificada mediante `0` y `1`, por lo que sus estadísticas deben interpretarse de acuerdo con dicha codificación y no como una magnitud cuantitativa continua.

### 4.2 Variables categóricas

In [33]:
#Resumen descriptivo de las variables categóricas
data.describe(include="object")

,customerID,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Churn
count,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043
unique,7043,2,2,2,2,3,3,3,3,3,3,3,3,3,2,4,2
top,3186-AJIEK,Male,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,No
freq,1,3555,3641,4933,6361,3390,3096,3498,3088,3095,3473,2810,2785,3875,4171,2365,5174


El resumen de las variables categóricas permite identificar la cantidad de categorías presentes y aquellas con mayor frecuencia.

La variable objetivo `Churn` presenta dos categorías, siendo `No` la más frecuente con **5.174 registros**.

Por otra parte, `customerID` presenta **7.043 valores únicos para 7.043 observaciones**, lo que es consistente con su función como identificador individual de cada cliente.

Las demás variables categóricas presentan un número reducido de categorías, las cuales serán analizadas posteriormente en relación con `Churn` para identificar diferencias en las tasas de abandono entre distintos grupos de clientes.

## 5. Análisis Exploratorio de Datos (EDA)

### 5.1 Magnitud del abandono de clientes

El primer análisis busca cuantificar la magnitud del abandono dentro del conjunto de datos, considerando tanto la cantidad de clientes como su proporción respecto del total.

In [34]:
# Cálculo de la cantidad y porcentaje de clientes según Churn
churn_summary = (
    data["Churn"]
    .value_counts()
    .rename_axis("Churn")
    .reset_index(name="Clientes")
)

churn_summary["Porcentaje"] = (
    churn_summary["Clientes"] / len(data) * 100
)

churn_summary

,Churn,Clientes,Porcentaje
0,No,5174,73.463013
1,Yes,1869,26.536987


In [35]:
#Etiquetas comunicacionales para la visualización
churn_summary["Estado"] = churn_summary["Churn"].map({
    "No": "Permanece",
    "Yes": "Abandona"
})

#Orden de presentación
churn_summary["Estado"] = pd.Categorical(
    churn_summary["Estado"],
    categories=["Permanece", "Abandona"],
    ordered=True
)

churn_summary = churn_summary.sort_values("Estado")

In [36]:
#Visualización ejecutiva de la magnitud del Churn
fig = px.bar(
    churn_summary,
    x="Estado",
    y="Porcentaje",
    color="Estado",
    text="Porcentaje",
    custom_data=["Clientes"],
    title=(
        "<b>El 26,5 % de los clientes del dataset presenta abandono</b>"
        "<br><sup>1.869 de 7.043 clientes tienen Churn = Yes</sup>"
    ),
    labels={
        "Estado": "",
        "Porcentaje": "Porcentaje de clientes (%)"
    },
    color_discrete_map={
        "Permanece": "#4C78A8",
        "Abandona": "#E45756"
    }
)

fig.update_traces(
    texttemplate="%{text:.1f} %",
    textposition="outside",
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Clientes: %{customdata[0]:,.0f}<br>"
        "Porcentaje: %{y:.2f} %"
        "<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_white",
    showlegend=False,
    height=500,
    bargap=0.45,
    title_x=0.02,
    font=dict(size=14),
    margin=dict(t=110, l=70, r=40, b=60)
)

fig.update_yaxes(
    range=[0, 80],
    ticksuffix=" %",
    showgrid=True,
    zeroline=False
)

fig.show()

La tasa de abandono observada en el conjunto de datos alcanza aproximadamente **26,54 %**, equivalente a **1.869 de los 7.043 clientes analizados**. En contraste, **5.174 clientes (73,46 %)** permanecen en la compañía.

Esto significa que aproximadamente **1 de cada 4 clientes del dataset presenta abandono**, evidenciando una proporción relevante sobre la cual resulta pertinente profundizar para identificar qué características diferencian a quienes abandonan de quienes permanecen.

### 5.2 Distribución de la permanencia de los clientes

Se analiza la distribución de `tenure` para comprender cómo se distribuyen los meses de permanencia de los clientes y detectar concentraciones, dispersión y posibles comportamientos relevantes antes de relacionar esta variable con `Churn`.

In [37]:
#Estadísticos principales de tenure
q1_tenure = data["tenure"].quantile(0.25)
median_tenure = data["tenure"].median()
q3_tenure = data["tenure"].quantile(0.75)

print("Q1:", q1_tenure)
print("Mediana:", median_tenure)
print("Q3:", q3_tenure)

Q1: 9.0
Mediana: 29.0
Q3: 55.0


In [38]:
#Distribución de los meses de permanencia de los clientes
fig = px.histogram(
    data,
    x="tenure",
    title=(
        "<b>La mitad de los clientes presenta entre 9 y 55 meses de permanencia</b>"
        "<br><sup>La mediana de permanencia de la cartera es de 29 meses</sup>"
    ),
    labels={
        "tenure": "Permanencia del cliente (meses)"
    }
)

#Intervalos de 6 meses para facilitar la lectura
fig.update_traces(
    xbins=dict(
        start=0,
        end=73,
        size=6
    ),
    marker_color="#2F5D8C",
    marker_line_width=0,
    opacity=0.9,
    hovertemplate=(
        "Permanencia: %{x}<br>"
        "Clientes: %{y}"
        "<extra></extra>"
    )
)

#Zona correspondiente al 50 % central de los clientes
fig.add_vrect(
    x0=q1_tenure,
    x1=q3_tenure,
    fillcolor="#2F5D8C",
    opacity=0.08,
    line_width=0
)

#Línea correspondiente a la mediana
fig.add_vline(
    x=median_tenure,
    line_width=2,
    line_dash="dash",
    line_color="#D95F59",
    annotation_text="Mediana: 29 meses",
    annotation_position="top"
)

fig.update_layout(
    template="plotly_white",
    height=520,
    bargap=0.06,
    showlegend=False,
    title_x=0.02,
    font=dict(
        family="Arial",
        size=14,
        color="#263238"
    ),
    margin=dict(
        t=110,
        l=70,
        r=40,
        b=70
    ),
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.update_xaxes(
    title="Permanencia del cliente (meses)",
    tickmode="linear",
    dtick=12,
    range=[0, 73],
    showgrid=False,
    zeroline=False
)

fig.update_yaxes(
    title="Cantidad de clientes",
    gridcolor="#E9EEF3",
    zeroline=False
)

fig.show()

La permanencia de los clientes presenta una amplitud considerable, con valores comprendidos entre **0 y 72 meses**.

La mediana se sitúa en **29 meses**, mientras que el 50 % central de los clientes presenta una permanencia comprendida entre **9 y 55 meses**. La media de aproximadamente **32,37 meses** y una desviación estándar de **24,56 meses** evidencian una cartera heterogénea en términos de antigüedad.

La distribución muestra presencia tanto de clientes con pocos meses de permanencia como de clientes con relaciones de mayor duración. Sin embargo, este análisis por sí solo no permite determinar si la permanencia está asociada al abandono, por lo que resulta necesario comparar posteriormente `tenure` entre las categorías de `Churn`.

### 5.3 Permanencia de los clientes según abandono

Se compara la distribución de `tenure` entre los clientes que permanecen y aquellos que abandonan la compañía, con el objetivo de determinar si existen diferencias relevantes en su antigüedad.

In [39]:
#Resumen estadístico de tenure según Churn
tenure_churn_summary = (
    data.groupby("Churn")["tenure"]
    .agg(
        Clientes="count",
        Media="mean",
        Mediana="median",
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75)
    )
    .reset_index()
)

#Etiquetas más comprensibles para la presentación
tenure_churn_summary["Estado"] = tenure_churn_summary["Churn"].map({
    "No": "Permanece",
    "Yes": "Abandona"
})

tenure_churn_summary[
    ["Estado", "Clientes", "Media", "Mediana", "Q1", "Q3"]
].round(2)

,Estado,Clientes,Media,Mediana,Q1,Q3
0,Permanece,5174,37.57,38.0,15.0,61.0
1,Abandona,1869,17.98,10.0,2.0,29.0


In [40]:
# Preparación de los grupos
tenure_permanece = data.loc[data["Churn"] == "No", "tenure"]
tenure_abandona = data.loc[data["Churn"] == "Yes", "tenure"]

# Estadísticos principales
mediana_permanece = tenure_permanece.median()
mediana_abandona = tenure_abandona.median()

q1_permanece = tenure_permanece.quantile(0.25)
q3_permanece = tenure_permanece.quantile(0.75)

q1_abandona = tenure_abandona.quantile(0.25)
q3_abandona = tenure_abandona.quantile(0.75)

diferencia_mediana = mediana_permanece - mediana_abandona

# Creación de la figura
fig = go.Figure()

# Clientes que permanecen
fig.add_trace(
    go.Box(
        y=tenure_permanece,
        name="Permanece",
        boxpoints="outliers",
        fillcolor="rgba(56, 114, 170, 0.22)",
        line=dict(
            color="#326FA8",
            width=3
        ),
        marker=dict(
            color="#326FA8",
            size=6,
            opacity=0.65
        ),
        whiskerwidth=0.7,
        width=0.48,
        hoverlabel=dict(
            bgcolor="white",
            bordercolor="#CBD5E1",
            font=dict(
                color="#17212B",
                size=13
            )
        )
    )
)

# Clientes que abandonan
fig.add_trace(
    go.Box(
        y=tenure_abandona,
        name="Abandona",
        boxpoints="outliers",
        fillcolor="rgba(225, 105, 97, 0.24)",
        line=dict(
            color="#D95F59",
            width=3
        ),
        marker=dict(
            color="#D95F59",
            size=6,
            opacity=0.65
        ),
        whiskerwidth=0.7,
        width=0.48,
        hoverlabel=dict(
            bgcolor="white",
            bordercolor="#CBD5E1",
            font=dict(
                color="#17212B",
                size=13
            )
        )
    )
)

# Mediana de quienes permanecen
fig.add_annotation(
    x="Permanece",
    y=mediana_permanece,
    text=f"<b>{mediana_permanece:.0f} meses</b>",
    showarrow=False,
    xshift=72,
    font=dict(
        size=15,
        color="#326FA8"
    ),
    bgcolor="rgba(255,255,255,0.94)",
    bordercolor="#326FA8",
    borderwidth=1.5,
    borderpad=7
)

# Mediana de quienes abandonan
fig.add_annotation(
    x="Abandona",
    y=mediana_abandona,
    text=f"<b>{mediana_abandona:.0f} meses</b>",
    showarrow=False,
    xshift=72,
    font=dict(
        size=15,
        color="#D95F59"
    ),
    bgcolor="rgba(255,255,255,0.94)",
    bordercolor="#D95F59",
    borderwidth=1.5,
    borderpad=7
)

# Mensaje visual central con la diferencia
fig.add_annotation(
    x=0.5,
    y=1.04,
    xref="paper",
    yref="paper",
    text=f"<b>↓ {diferencia_mediana:.0f} meses de diferencia en la mediana</b>",
    showarrow=False,
    font=dict(
        size=15,
        color="#334155"
    ),
    bgcolor="#F4F7FA",
    bordercolor="#D9E1E8",
    borderwidth=1,
    borderpad=8
)

# Diseño general
fig.update_layout(
    title=dict(
        text=(
            "<b>Los clientes que abandonan presentan una permanencia mediana "
            "28 meses menor</b>"
            "<br>"
            "<sup>La mediana es de 10 meses entre quienes abandonan, "
            "frente a 38 meses entre quienes permanecen</sup>"
        ),
        x=0.02,
        xanchor="left"
    ),
    template="plotly_white",
    height=620,
    showlegend=False,
    font=dict(
        family="Arial",
        size=14,
        color="#263238"
    ),
    margin=dict(
        t=145,
        l=80,
        r=100,
        b=75
    ),
    plot_bgcolor="#FCFDFE",
    paper_bgcolor="white",
    hoverlabel=dict(
        bgcolor="white",
        bordercolor="#CBD5E1",
        font=dict(
            color="#17212B",
            size=13
        )
    )
)

fig.update_xaxes(
    title="",
    showgrid=False,
    tickfont=dict(
        size=16,
        color="#334155"
    ),
    showline=False
)

fig.update_yaxes(
    title="Permanencia del cliente (meses)",
    range=[0, 76],
    dtick=12,
    gridcolor="#E7ECF1",
    gridwidth=1,
    zeroline=False,
    tickfont=dict(
        size=12,
        color="#64748B"
    )
)

fig.show()

La permanencia de los clientes presenta diferencias importantes según su condición de abandono.

Los clientes que permanecen en la compañía presentan una mediana de **38 meses**, mientras que aquellos que abandonan presentan una mediana de solamente **10 meses**, una diferencia de **28 meses**.

Además, el 50 % central de los clientes que abandonan se concentra entre **2 y 29 meses** de permanencia, mientras que entre quienes permanecen se encuentra entre **15 y 61 meses**.

Estos resultados evidencian una asociación entre menor antigüedad y `Churn = Yes`. En particular, el primer cuartil del grupo que abandona se sitúa en **2 meses**, lo que indica presencia relevante de abandono entre clientes de muy baja antigüedad.

El boxplot también identifica posibles valores atípicos superiores entre los clientes que abandonan. Estos valores se encuentran dentro del rango general válido de `tenure`, por lo que no existe evidencia suficiente para considerarlos errores o eliminarlos.

### 5.4 Tasa de abandono según tipo de contrato

Se analiza la tasa de `Churn` dentro de cada tipo de contrato con el objetivo de identificar si existen diferencias relevantes en la proporción de clientes que abandonan la compañía.

Se utilizan tasas porcentuales en lugar de cantidades absolutas, ya que los distintos tipos de contrato contienen diferentes cantidades de clientes y una comparación directa de frecuencias podría resultar engañosa.

In [41]:
#Cálculo de la tasa de abandono dentro de cada tipo de contrato
contract_churn = (
    data.groupby("Contract")["Churn"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("Porcentaje")
    .reset_index()
)

#Selección de la categoría Churn = Yes
contract_churn_yes = contract_churn[
    contract_churn["Churn"] == "Yes"
].copy()

#Orden de los contratos para facilitar la comparación
contract_churn_yes["Contract"] = pd.Categorical(
    contract_churn_yes["Contract"],
    categories=["Month-to-month", "One year", "Two year"],
    ordered=True
)

contract_churn_yes = contract_churn_yes.sort_values("Contract")

contract_churn_yes

,Contract,Churn,Porcentaje
1,Month-to-month,Yes,42.709677
3,One year,Yes,11.269518
5,Two year,Yes,2.831858


In [42]:
# Etiquetas más claras para presentación
contract_churn_yes["Contrato"] = contract_churn_yes["Contract"].map({
    "Month-to-month": "Mensual",
    "One year": "1 año",
    "Two year": "2 años"
})

# Color destacado para el segmento con mayor tasa de abandono
contract_churn_yes["Grupo"] = contract_churn_yes["Contrato"].apply(
    lambda x: "Mayor abandono" if x == "Mensual" else "Otros contratos"
)

fig = px.bar(
    contract_churn_yes,
    x="Porcentaje",
    y="Contrato",
    orientation="h",
    color="Grupo",
    text="Porcentaje",
    title=(
        "<b>El abandono alcanza 42,7 % entre los clientes con contrato mensual</b>"
        "<br><sup>La tasa disminuye a 11,3 % en contratos de 1 año y a 2,8 % en contratos de 2 años</sup>"
    ),
    labels={
        "Porcentaje": "Tasa de abandono (%)",
        "Contrato": ""
    },
    color_discrete_map={
        "Mayor abandono": "#D95F59",
        "Otros contratos": "#8FA7BF"
    }
)

fig.update_traces(
    texttemplate="%{text:.1f} %",
    textposition="outside",
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Tasa de abandono: %{x:.2f} %"
        "<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_white",
    height=500,
    showlegend=False,
    title_x=0.02,
    font=dict(
        family="Arial",
        size=14,
        color="#263238"
    ),
    margin=dict(
        t=120,
        l=100,
        r=80,
        b=70
    ),
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.update_xaxes(
    title="Tasa de abandono (%)",
    range=[0, 50],
    ticksuffix=" %",
    gridcolor="#E9EEF3",
    zeroline=False
)

fig.update_yaxes(
    title="",
    categoryorder="array",
    categoryarray=["2 años", "1 año", "Mensual"],
    showgrid=False
)

fig.show()

La tasa de abandono presenta diferencias marcadas según el tipo de contrato.

Los clientes con contrato mensual (`Month-to-month`) presentan una tasa de `Churn` de aproximadamente **42,71 %**, mientras que esta disminuye a **11,27 %** entre los clientes con contrato de un año y a solamente **2,83 %** entre quienes poseen contrato de dos años.

La diferencia entre los contratos mensuales y los contratos de dos años alcanza aproximadamente **39,88 puntos porcentuales**.

Estos resultados evidencian una fuerte asociación entre el tipo de contrato y el abandono de clientes dentro del conjunto de datos. En particular, los clientes con contratos de menor duración presentan una mayor proporción de abandono.

El análisis es exploratorio, por lo que no permite concluir que el tipo de contrato cause directamente el abandono.

### 5.5 Cargos mensuales según abandono

Se compara la distribución de `MonthlyCharges` entre los clientes que permanecen y aquellos que abandonan la compañía, con el objetivo de identificar si existen diferencias relevantes en los cargos mensuales asociados a `Churn`.

In [43]:
#Resumen estadístico de MonthlyCharges según Churn
monthly_charges_summary = (
    data.groupby("Churn")["MonthlyCharges"]
    .agg(
        Clientes="count",
        Media="mean",
        Mediana="median",
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75)
    )
    .reset_index()
)

monthly_charges_summary["Estado"] = monthly_charges_summary["Churn"].map({
    "No": "Permanece",
    "Yes": "Abandona"
})

monthly_charges_summary[
    ["Estado", "Clientes", "Media", "Mediana", "Q1", "Q3"]
].round(2)

,Estado,Clientes,Media,Mediana,Q1,Q3
0,Permanece,5174,61.27,64.43,25.10,88.4
1,Abandona,1869,74.44,79.65,56.15,94.2


In [44]:
# Preparación de etiquetas para la visualización
monthly_plot = data.copy()

monthly_plot["Estado"] = monthly_plot["Churn"].map({
    "No": "Permanece",
    "Yes": "Abandona"
})

# Medianas utilizadas para destacar el hallazgo
mediana_mensual_permanece = monthly_plot.loc[
    monthly_plot["Estado"] == "Permanece",
    "MonthlyCharges"
].median()

mediana_mensual_abandona = monthly_plot.loc[
    monthly_plot["Estado"] == "Abandona",
    "MonthlyCharges"
].median()

diferencia_mensual = (
    mediana_mensual_abandona - mediana_mensual_permanece
)

# Boxplot vertical
fig = px.box(
    monthly_plot,
    x="Estado",
    y="MonthlyCharges",
    color="Estado",
    points="outliers",
    category_orders={
        "Estado": ["Permanece", "Abandona"]
    },
    title=(
        "<b>Quienes abandonan presentan cargos mensuales medianos más altos</b>"
        f"<br><sup>La diferencia mediana alcanza aproximadamente "
        f"{diferencia_mensual:.2f} unidades monetarias</sup>"
    ),
    labels={
        "Estado": "",
        "MonthlyCharges": "Cargo mensual"
    }
)

fig.update_traces(
    selector=dict(name="Permanece"),
    fillcolor="#DCEBFA",
    line=dict(
        color="#2F5D8C",
        width=2
    ),
    marker=dict(
        color="#2F5D8C",
        size=6,
        opacity=0.70
    )
)

fig.update_traces(
    selector=dict(name="Abandona"),
    fillcolor="#FCE1DE",
    line=dict(
        color="#D95F59",
        width=2
    ),
    marker=dict(
        color="#D95F59",
        size=6,
        opacity=0.70
    )
)

fig.add_annotation(
    x="Permanece",
    y=mediana_mensual_permanece,
    text=f"<b>{mediana_mensual_permanece:.2f}</b>",
    showarrow=False,
    yshift=24,
    bgcolor="white",
    bordercolor="#2F5D8C",
    borderwidth=1,
    borderpad=6,
    font=dict(
        color="#2F5D8C",
        size=13
    )
)

fig.add_annotation(
    x="Abandona",
    y=mediana_mensual_abandona,
    text=f"<b>{mediana_mensual_abandona:.2f}</b>",
    showarrow=False,
    yshift=24,
    bgcolor="white",
    bordercolor="#D95F59",
    borderwidth=1,
    borderpad=6,
    font=dict(
        color="#D95F59",
        size=13
    )
)

fig.update_traces(
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Cargo mensual: %{y:.2f}"
        "<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_white",
    height=560,
    showlegend=False,
    title_x=0.02,
    font=dict(
        family="Arial",
        size=14,
        color="#263238"
    ),
    hoverlabel=dict(
        bgcolor="white",
        bordercolor="#D1D5DB",
        font=dict(
            color="#111827",
            size=13,
            family="Arial"
        )
    ),
    margin=dict(
        t=120,
        l=80,
        r=50,
        b=70
    ),
    plot_bgcolor="white",
    paper_bgcolor="white"
)

fig.update_xaxes(
    title="",
    showgrid=False,
    tickfont=dict(size=15)
)

fig.update_yaxes(
    title="Cargo mensual",
    gridcolor="#E9EEF3",
    zeroline=False
)

fig.show()

Los clientes que abandonan presentan cargos mensuales superiores a quienes permanecen.

La mediana de `MonthlyCharges` alcanza aproximadamente **79,65** entre los clientes con `Churn = Yes`, frente a **64,43** entre quienes permanecen, lo que representa una diferencia cercana a **15,22 unidades monetarias**.

Asimismo, el 50 % central de los clientes que abandonan presenta cargos mensuales entre **56,15 y 94,20**, mientras que entre quienes permanecen este intervalo se sitúa entre **25,10 y 88,40**.

El análisis agregado muestra que los clientes que abandonan presentan cargos mensuales superiores a quienes permanecen. Sin embargo, esta comparación por sí sola no permite determinar si `MonthlyCharges` mantiene la misma relación con `Churn` al considerar otras características de los clientes.

Por esta razón, posteriormente se profundiza este hallazgo incorporando el tipo de servicio de Internet, con el objetivo de evitar una interpretación aislada de los cargos mensuales.

### 5.6 Tasa de abandono según servicio de Internet

Se analiza la tasa de `Churn` dentro de cada tipo de servicio de Internet, con el objetivo de identificar si existen diferencias relevantes en la proporción de abandono entre los distintos segmentos de clientes.

In [45]:
#Cálculo de la tasa de abandono según tipo de servicio de Internet
internet_churn = (
    data.groupby("InternetService")["Churn"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("Porcentaje")
    .reset_index()
)

#Selección de la categoría Churn = Yes
internet_churn_yes = internet_churn[
    internet_churn["Churn"] == "Yes"
].copy()

internet_churn_yes

,InternetService,Churn,Porcentaje
1,DSL,Yes,18.959108
3,Fiber optic,Yes,41.892765
5,No,Yes,7.404980


In [46]:
# Etiquetas más claras para presentación
internet_churn_yes["Servicio"] = internet_churn_yes["InternetService"].map({
    "Fiber optic": "Fibra óptica",
    "DSL": "DSL",
    "No": "Sin Internet"
})

# Orden de presentación
orden_servicios = ["Fibra óptica", "DSL", "Sin Internet"]

internet_churn_yes["Servicio"] = pd.Categorical(
    internet_churn_yes["Servicio"],
    categories=orden_servicios,
    ordered=True
)

internet_churn_yes = internet_churn_yes.sort_values("Servicio")

# Tasa global de abandono
tasa_global_churn = (
    data["Churn"].eq("Yes").mean() * 100
)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=internet_churn_yes["Servicio"],
        y=internet_churn_yes["Porcentaje"],
        text=[
            f"{valor:.1f} %"
            for valor in internet_churn_yes["Porcentaje"]
        ],
        textposition="outside",
        textfont=dict(
            size=15,
            color="#1F2937"
        ),
        marker=dict(
            color=[
                "#D95F59",
                "#5B8DB8",
                "#A8B8C6"
            ],
            line=dict(
                color="white",
                width=1.5
            )
        ),
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Tasa de abandono: %{y:.2f} %"
            "<extra></extra>"
        )
    )
)

# Línea de referencia: tasa global de churn
fig.add_hline(
    y=tasa_global_churn,
    line_dash="dash",
    line_width=2,
    line_color="#6B7280",
    annotation_text=f"Tasa global: {tasa_global_churn:.1f} %",
    annotation_position="top right",
    annotation_font=dict(
        color="#4B5563",
        size=12
    )
)

fig.update_layout(
    title=dict(
        text=(
            "<b>La fibra óptica presenta la mayor tasa de abandono: 41,9 %</b>"
            "<br>"
            "<sup>Supera en 15,4 puntos porcentuales la tasa global de churn del dataset</sup>"
        ),
        x=0.02,
        xanchor="left"
    ),
    template="plotly_white",
    height=560,
    showlegend=False,
    font=dict(
        family="Arial",
        size=14,
        color="#263238"
    ),
    margin=dict(
        t=125,
        l=75,
        r=45,
        b=70
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    hoverlabel=dict(
        bgcolor="white",
        bordercolor="#D1D5DB",
        font=dict(
            color="#111827",
            size=13,
            family="Arial"
        )
    )
)

fig.update_xaxes(
    title="",
    showgrid=False,
    tickfont=dict(
        size=14,
        color="#374151"
    )
)

fig.update_yaxes(
    title="Tasa de abandono (%)",
    range=[0, 50],
    dtick=10,
    ticksuffix=" %",
    gridcolor="#E9EEF3",
    zeroline=False
)

fig.show()

La tasa de abandono presenta diferencias relevantes según el tipo de servicio de Internet contratado.

Los clientes con **fibra óptica** presentan la mayor tasa de abandono, con aproximadamente **41,89 %**, superando en cerca de **15,35 puntos porcentuales** la tasa global de `Churn` del conjunto de datos.

En contraste, los clientes con servicio **DSL** presentan una tasa de abandono de aproximadamente **18,96 %**, mientras que aquellos sin servicio de Internet alcanzan aproximadamente **7,40 %**.

Este resultado evidencia una asociación entre el tipo de servicio de Internet y `Churn`. Sin embargo, no permite concluir que la fibra óptica sea la causa directa del abandono, ya que otras características asociadas a estos clientes podrían estar influyendo en el patrón observado.

### 5.7 Tasa de abandono según método de pago

Se analiza la tasa de `Churn` dentro de cada método de pago con el objetivo de identificar si existen diferencias relevantes en la proporción de abandono entre los distintos segmentos de clientes.

In [47]:
# Cálculo de la tasa de abandono según método de pago
payment_churn = (
    data.groupby("PaymentMethod")["Churn"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("Porcentaje")
    .reset_index()
)

# Selección de Churn = Yes
payment_churn_yes = payment_churn[
    payment_churn["Churn"] == "Yes"
].copy()

# Orden descendente para facilitar la comparación
payment_churn_yes = payment_churn_yes.sort_values(
    "Porcentaje",
    ascending=False
)

payment_churn_yes

,PaymentMethod,Churn,Porcentaje
5,Electronic check,Yes,45.285412
7,Mailed check,Yes,19.106700
1,Bank transfer (automatic),Yes,16.709845
3,Credit card (automatic),Yes,15.243101


In [48]:
# Etiquetas más claras para presentación
payment_churn_yes["Metodo"] = payment_churn_yes["PaymentMethod"].map({
    "Electronic check": "Cheque electrónico",
    "Mailed check": "Cheque por correo",
    "Bank transfer (automatic)": "Transferencia automática",
    "Credit card (automatic)": "Tarjeta automática"
})

In [49]:
# Tasa global de abandono
tasa_global_churn = data["Churn"].eq("Yes").mean() * 100

# Orden visual de menor a mayor para que la mayor tasa quede arriba
payment_plot = payment_churn_yes.sort_values(
    "Porcentaje",
    ascending=True
)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=payment_plot["Porcentaje"],
        y=payment_plot["Metodo"],
        orientation="h",
        text=[
            f"{valor:.1f} %"
            for valor in payment_plot["Porcentaje"]
        ],
        textposition="outside",
        textfont=dict(
            size=14,
            color="#1F2937"
        ),
        marker=dict(
            color=[
                "#A9BAC8",
                "#93ABC0",
                "#7899B6",
                "#D95F59"
            ],
            line=dict(
                color="white",
                width=1.5
            )
        ),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Tasa de abandono: %{x:.2f} %"
            "<extra></extra>"
        )
    )
)

# Benchmark de la tasa global
fig.add_vline(
    x=tasa_global_churn,
    line_dash="dash",
    line_width=2,
    line_color="#6B7280"
)

fig.add_annotation(
    x=tasa_global_churn,
    y=1.04,
    xref="x",
    yref="paper",
    text=f"Tasa global: {tasa_global_churn:.1f} %",
    showarrow=False,
    font=dict(
        size=12,
        color="#4B5563"
    ),
    bgcolor="white"
)

fig.update_layout(
    title=dict(
        text=(
            "<b>El cheque electrónico concentra la mayor tasa de abandono: 45,3 %</b>"
            "<br>"
            "<sup>Supera en 18,7 puntos porcentuales la tasa global del dataset</sup>"
        ),
        x=0.02,
        xanchor="left"
    ),
    template="plotly_white",
    height=560,
    showlegend=False,
    font=dict(
        family="Arial",
        size=14,
        color="#263238"
    ),
    margin=dict(
        t=125,
        l=190,
        r=75,
        b=70
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    hoverlabel=dict(
        bgcolor="white",
        bordercolor="#D1D5DB",
        font=dict(
            color="#111827",
            size=13
        )
    )
)

fig.update_xaxes(
    title="Tasa de abandono (%)",
    range=[0, 50],
    dtick=10,
    ticksuffix=" %",
    gridcolor="#E9EEF3",
    zeroline=False
)

fig.update_yaxes(
    title="",
    showgrid=False,
    tickfont=dict(
        size=13,
        color="#374151"
    )
)

fig.show()

La tasa de abandono presenta diferencias relevantes según el método de pago utilizado.

Los clientes que utilizan **cheque electrónico** presentan la mayor tasa de `Churn`, con aproximadamente **45,29 %**, superando en cerca de **18,75 puntos porcentuales** la tasa global de abandono del conjunto de datos.

En contraste, las tasas de abandono observadas en cheque por correo, transferencia bancaria automática y tarjeta de crédito automática son aproximadamente **19,11 %**, **16,71 %** y **15,24 %**, respectivamente.

Este resultado evidencia una asociación entre el método de pago y el abandono. Sin embargo, no permite concluir que el cheque electrónico cause directamente el `Churn`, ya que este segmento puede presentar simultáneamente otras características relacionadas con una mayor tasa de abandono.

### 5.8 Profundización de la relación entre contrato, permanencia y abandono

Los análisis anteriores mostraron una mayor tasa de abandono entre clientes con contratos mensuales y una menor permanencia entre quienes presentan `Churn = Yes`.

Para determinar si ambos hallazgos están relacionados, se analiza conjuntamente el tipo de contrato, la permanencia del cliente y la tasa de abandono.

In [50]:
# Copia destinada al análisis multivariado
contract_tenure = data.copy()

# Segmentación de tenure utilizando los cuartiles observados
contract_tenure["SegmentoTenure"] = pd.cut(
    contract_tenure["tenure"],
    bins=[-1, 9, 29, 55, 72],
    labels=[
        "0–9 meses",
        "10–29 meses",
        "30–55 meses",
        "56–72 meses"
    ]
)

In [51]:
# Cantidad de clientes según contrato, segmento de tenure y Churn
contract_tenure_churn = (
    contract_tenure
    .groupby(
        ["Contract", "SegmentoTenure", "Churn"],
        observed=True
    )
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Total de clientes de cada segmento
contract_tenure_churn["Clientes"] = (
    contract_tenure_churn["No"]
    + contract_tenure_churn["Yes"]
)

# Tasa de abandono dentro de cada segmento
contract_tenure_churn["TasaChurn"] = (
    contract_tenure_churn["Yes"]
    / contract_tenure_churn["Clientes"]
    * 100
)

contract_tenure_churn

Churn,Contract,SegmentoTenure,No,Yes,Clientes,TasaChurn
0,Month-to-month,0–9 meses,816,916,1732,52.886836
1,Month-to-month,10–29 meses,763,457,1220,37.459016
2,Month-to-month,30–55 meses,489,239,728,32.829670
3,Month-to-month,56–72 meses,152,43,195,22.051282
4,One year,0–9 meses,66,7,73,9.589041
5,One year,10–29 meses,328,28,356,7.865169
6,One year,30–55 meses,532,74,606,12.211221
7,One year,56–72 meses,381,57,438,13.013699
8,Two year,0–9 meses,49,0,49,0.000000
9,Two year,10–29 meses,138,1,139,0.719424


In [52]:
# Etiquetas de contrato para presentación
contract_tenure_churn["Contrato"] = (
    contract_tenure_churn["Contract"].map({
        "Month-to-month": "Mensual",
        "One year": "1 año",
        "Two year": "2 años"
    })
)

# Matriz de tasas de abandono
heatmap_rate = (
    contract_tenure_churn
    .pivot(
        index="Contrato",
        columns="SegmentoTenure",
        values="TasaChurn"
    )
    .reindex(["Mensual", "1 año", "2 años"])
)

# Matriz de cantidad de clientes
heatmap_n = (
    contract_tenure_churn
    .pivot(
        index="Contrato",
        columns="SegmentoTenure",
        values="Clientes"
    )
    .reindex(["Mensual", "1 año", "2 años"])
)

In [53]:
fig = go.Figure()

fig.add_trace(
    go.Heatmap(
        z=heatmap_rate.values,
        x=heatmap_rate.columns.astype(str),
        y=heatmap_rate.index,

        customdata=heatmap_n.values,

        colorscale=[
            [0.00, "#F7FAFC"],
            [0.20, "#E7EEF5"],
            [0.40, "#F6DDD9"],
            [0.65, "#E99A91"],
            [1.00, "#C94F49"]
        ],

        zmin=0,
        zmax=55,

        colorbar=dict(
            title="Churn (%)",
            ticksuffix=" %",
            thickness=14,
            len=0.82,
            outlinewidth=0
        ),

        hovertemplate=(
            "<b>%{y}</b><br>"
            "Permanencia: %{x}<br>"
            "Tasa de abandono: %{z:.2f} %<br>"
            "Clientes del segmento: %{customdata:,.0f}"
            "<extra></extra>"
        )
    )
)

# Texto dentro de cada celda
for fila, contrato in enumerate(heatmap_rate.index):
    for columna, segmento in enumerate(heatmap_rate.columns):

        tasa = heatmap_rate.iloc[fila, columna]
        clientes = heatmap_n.iloc[fila, columna]

        # Contraste del texto según intensidad de la celda
        color_texto = "white" if tasa >= 35 else "#263238"

        fig.add_annotation(
            x=str(segmento),
            y=contrato,
            text=(
                f"<b>{tasa:.1f} %</b>"
                f"<br><span style='font-size:11px'>n = {clientes:,.0f}</span>"
            ),
            showarrow=False,
            font=dict(
                size=14,
                color=color_texto
            )
        )

fig.update_layout(
    title=dict(
        text=(
            "<b>El mayor abandono se concentra en clientes mensuales "
            "con baja permanencia</b>"
            "<br>"
            "<sup>La tasa alcanza 52,9 % durante los primeros 9 meses "
            "y disminuye a 22,1 % después de 55 meses</sup>"
        ),
        x=0.02,
        xanchor="left"
    ),

    template="plotly_white",
    height=560,

    font=dict(
        family="Arial",
        size=14,
        color="#263238"
    ),

    margin=dict(
        t=125,
        l=100,
        r=80,
        b=85
    ),

    paper_bgcolor="white",

    hoverlabel=dict(
        bgcolor="white",
        bordercolor="#D1D5DB",
        font=dict(
            color="#111827",
            size=13
        )
    )
)

fig.update_xaxes(
    title="<b>Permanencia del cliente</b>",
    side="bottom",
    tickfont=dict(size=13),
    showgrid=False
)

fig.update_yaxes(
    title="<b>Tipo de contrato</b>",
    tickfont=dict(size=14),
    showgrid=False,
    autorange="reversed"
)

fig.show()

El análisis conjunto de `Contract`, `tenure` y `Churn` muestra que la asociación entre permanencia y abandono no se comporta de la misma forma para todos los tipos de contrato.

Entre los clientes con contrato mensual, la tasa de abandono alcanza aproximadamente **52,89 %** durante los primeros 9 meses de permanencia y disminuye progresivamente hasta **22,05 %** entre quienes presentan más de 55 meses.

En contraste, los contratos de un año mantienen tasas de abandono considerablemente menores, aproximadamente entre **7,87 % y 13,01 %**, mientras que los contratos de dos años presentan tasas inferiores a **3,21 %** en todos los segmentos analizados.

El segmento con mayor tasa observada corresponde a clientes con **contrato mensual y hasta 9 meses de permanencia**, donde **916 de 1.732 clientes (52,89 %)** presentan `Churn = Yes`.

Este resultado profundiza los hallazgos anteriores: la menor permanencia se encuentra asociada a una mayor tasa de abandono principalmente dentro del segmento de contratos mensuales, por lo que `tenure` y `Contract` no deben interpretarse de manera aislada.

### 5.9 Profundización de la relación entre servicio de Internet, cargos mensuales y abandono

El análisis agregado mostró que los clientes que abandonan presentan cargos mensuales medianos superiores a quienes permanecen. Paralelamente, los clientes con fibra óptica presentan una mayor tasa de abandono.

Para determinar si ambos hallazgos se encuentran relacionados, se comparan los cargos mensuales entre clientes que permanecen y abandonan dentro de cada tipo de servicio de Internet.

In [54]:
# Resumen de MonthlyCharges según servicio de Internet y Churn
monthly_service_summary = (
    data.groupby(["InternetService", "Churn"])["MonthlyCharges"]
    .agg(
        Clientes="count",
        Media="mean",
        Mediana="median"
    )
    .reset_index()
)

# Etiquetas para presentación
monthly_service_summary["Servicio"] = (
    monthly_service_summary["InternetService"].map({
        "No": "Sin Internet",
        "DSL": "DSL",
        "Fiber optic": "Fibra óptica"
    })
)

monthly_service_summary["Estado"] = (
    monthly_service_summary["Churn"].map({
        "No": "Permanece",
        "Yes": "Abandona"
    })
)

monthly_service_summary[
    ["Servicio", "Estado", "Clientes", "Media", "Mediana"]
].round(2)

,Servicio,Estado,Clientes,Media,Mediana
0,DSL,Permanece,1962,60.21,59.75
1,DSL,Abandona,459,49.08,49.25
2,Fibra óptica,Permanece,1799,93.93,94.80
3,Fibra óptica,Abandona,1297,88.13,87.55
4,Sin Internet,Permanece,1413,21.14,20.15
5,Sin Internet,Abandona,113,20.37,20.00


In [55]:
# Distribución del servicio de Internet dentro de cada categoría de Churn
internet_composition = (
    data.groupby("Churn")["InternetService"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("Porcentaje")
    .reset_index()
)

internet_composition["Estado"] = internet_composition["Churn"].map({
    "No": "Permanece",
    "Yes": "Abandona"
})

internet_composition.round(2)

,Churn,InternetService,Porcentaje,Estado
0,No,DSL,37.92,Permanece
1,No,Fiber optic,34.77,Permanece
2,No,No,27.31,Permanece
3,Yes,Fiber optic,69.40,Abandona
4,Yes,DSL,24.56,Abandona
5,Yes,No,6.05,Abandona


In [56]:
# Orden de presentación de los servicios
orden_servicios = [
    "Sin Internet",
    "DSL",
    "Fibra óptica"
]

permanece_service = (
    monthly_service_summary[
        monthly_service_summary["Estado"] == "Permanece"
    ]
    .set_index("Servicio")
    .reindex(orden_servicios)
    .reset_index()
)

abandona_service = (
    monthly_service_summary[
        monthly_service_summary["Estado"] == "Abandona"
    ]
    .set_index("Servicio")
    .reindex(orden_servicios)
    .reset_index()
)

fig = go.Figure()

# Clientes que permanecen
fig.add_trace(
    go.Bar(
        x=permanece_service["Servicio"],
        y=permanece_service["Mediana"],
        name="Permanece",
        marker=dict(
            color="#4F7CAC",
            line=dict(
                color="white",
                width=1
            )
        ),
        text=permanece_service["Mediana"],
        texttemplate="<b>%{text:.2f}</b>",
        textposition="outside",
        customdata=permanece_service[
            ["Clientes", "Media"]
        ].to_numpy(),
        hovertemplate=(
            "<b>%{x} · Permanece</b><br>"
            "Mediana mensual: %{y:.2f}<br>"
            "Media mensual: %{customdata[1]:.2f}<br>"
            "Clientes: %{customdata[0]:,.0f}"
            "<extra></extra>"
        )
    )
)

# Clientes que abandonan
fig.add_trace(
    go.Bar(
        x=abandona_service["Servicio"],
        y=abandona_service["Mediana"],
        name="Abandona",
        marker=dict(
            color="#E06B65",
            line=dict(
                color="white",
                width=1
            )
        ),
        text=abandona_service["Mediana"],
        texttemplate="<b>%{text:.2f}</b>",
        textposition="outside",
        customdata=abandona_service[
            ["Clientes", "Media"]
        ].to_numpy(),
        hovertemplate=(
            "<b>%{x} · Abandona</b><br>"
            "Mediana mensual: %{y:.2f}<br>"
            "Media mensual: %{customdata[1]:.2f}<br>"
            "Clientes: %{customdata[0]:,.0f}"
            "<extra></extra>"
        )
    )
)

fig.update_layout(
    title=dict(
        text=(
            "<b>El mayor cargo mensual global de quienes abandonan "
            "no se mantiene dentro de cada servicio</b>"
            "<br>"
            "<sup>El 69,4 % de los abandonos utiliza fibra óptica, "
            "el servicio con los cargos mensuales más altos</sup>"
        ),
        x=0.02,
        xanchor="left"
    ),

    barmode="group",
    bargap=0.28,
    bargroupgap=0.08,

    template="plotly_white",
    height=590,

    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.01,
        xanchor="right",
        x=1,
        title=""
    ),

    font=dict(
        family="Arial",
        size=14,
        color="#263238"
    ),

    margin=dict(
        t=145,
        l=80,
        r=50,
        b=75
    ),

    plot_bgcolor="white",
    paper_bgcolor="white",

    hoverlabel=dict(
        bgcolor="white",
        bordercolor="#D1D5DB",
        font=dict(
            color="#111827",
            size=13
        )
    ),

    uniformtext_minsize=12
)

fig.update_xaxes(
    title="",
    showgrid=False,
    tickfont=dict(
        size=14,
        color="#374151"
    )
)

fig.update_yaxes(
    title="Mediana del cargo mensual",
    range=[0, 110],
    dtick=20,
    gridcolor="#E9EEF3",
    zeroline=False,
    tickfont=dict(
        color="#64748B"
    )
)

fig.show()

El análisis conjunto modifica la interpretación obtenida inicialmente al evaluar `MonthlyCharges` de manera aislada.

Aunque globalmente los clientes que abandonan presentan una mediana de cargo mensual de **79,65**, frente a **64,43** entre quienes permanecen, esta diferencia no se mantiene al comparar clientes dentro del mismo tipo de servicio de Internet.

En fibra óptica, la mediana es de **87,55** entre quienes abandonan y **94,80** entre quienes permanecen. En DSL, las medianas corresponden a **49,25** y **59,75**, respectivamente, mientras que entre clientes sin servicio de Internet prácticamente no se observan diferencias.

Paralelamente, **69,40 % de los clientes que abandonan utiliza fibra óptica**, frente a **34,77 % de quienes permanecen**. Dado que este servicio presenta los cargos mensuales más elevados, la composición del grupo `Churn = Yes` contribuye a explicar la mayor mediana global observada inicialmente.

Por lo tanto, `MonthlyCharges` no debe interpretarse de forma aislada como una característica asociada independientemente a un mayor abandono. El tipo de servicio contratado resulta fundamental para contextualizar este hallazgo.

## 6. Preparación final del conjunto de datos

### 6.1 Exclusión del identificador único

La variable `customerID` fue utilizada durante la auditoría de calidad para verificar la unicidad de los registros. Se comprobó que existen 7.043 identificadores únicos para las 7.043 observaciones del conjunto de datos.

Sin embargo, `customerID` corresponde a un identificador individual y no representa una característica del comportamiento o relación comercial del cliente. Por esta razón, se excluye de la versión preparada del conjunto de datos destinada a análisis posteriores.

Esta decisión también responde a un criterio de minimización de datos, evitando conservar identificadores que no resultan necesarios para el propósito analítico.

In [57]:
# Creación del conjunto de datos preparado sin el identificador único
data_processed = data.drop(
    columns=["customerID"]
).copy()

# Verificación de dimensiones
print("Dataset original de trabajo:", data.shape)
print("Dataset preparado:", data_processed.shape)

Dataset original de trabajo: (7043, 21)
Dataset preparado: (7043, 20)


In [58]:
# Validación final del conjunto de datos preparado
print("Observaciones:", data_processed.shape[0])
print("Variables:", data_processed.shape[1])
print("Valores ausentes:", data_processed.isna().sum().sum())
print(
    "Filas repetidas tras excluir customerID:",
    data_processed.duplicated().sum()
)

Observaciones: 7043
Variables: 20
Valores ausentes: 0
Filas repetidas tras excluir customerID: 22


La versión preparada mantiene las **7.043 observaciones originales** y contiene **20 variables** luego de excluir `customerID`.

Tras excluir el identificador, se identificaron **22 filas repetidas respecto de una observación previa** en la versión preparada. Sin embargo, estos registros no corresponden a clientes duplicados, ya que la auditoría inicial confirmó que los **7.043 registros poseen `customerID` únicos** y que no existen filas completamente duplicadas en el conjunto original.

Por lo tanto, estas coincidencias representan clientes distintos que comparten exactamente las mismas características en las variables restantes. Los registros se mantienen para evitar eliminar observaciones válidas.

No se identifican valores ausentes después de las transformaciones realizadas. Tampoco se aplicaron codificaciones de variables categóricas ni otras transformaciones específicas de modelamiento, dado que el entrenamiento de modelos se encuentra fuera del alcance de esta evaluación.

In [59]:
# Exportación del conjunto de datos preparado
data_processed.to_csv(
    "/content/telco_customer_churn_processed.csv",
    index=False
)

## 7. Ética, privacidad y posibles sesgos

### 7.1 Evaluación de representatividad y variables personales

Se revisan características personales presentes en el conjunto de datos con el propósito de identificar diferencias de representación y posibles consideraciones de sesgo que deberían evaluarse antes de un eventual uso predictivo.

In [60]:
# Representación y tasa de Churn según gender
gender_ethics = (
    data.groupby("gender")["Churn"]
    .agg(
        Clientes="count",
        Abandonos=lambda x: (x == "Yes").sum()
    )
    .reset_index()
)

gender_ethics["Representacion"] = (
    gender_ethics["Clientes"] / len(data) * 100
)

gender_ethics["TasaChurn"] = (
    gender_ethics["Abandonos"]
    / gender_ethics["Clientes"]
    * 100
)

gender_ethics.round(2)

,gender,Clientes,Abandonos,Representacion,TasaChurn
0,Female,3488,939,49.52,26.92
1,Male,3555,930,50.48,26.16


In [61]:
# Representación y tasa de Churn según SeniorCitizen
senior_ethics = (
    data.groupby("SeniorCitizen")["Churn"]
    .agg(
        Clientes="count",
        Abandonos=lambda x: (x == "Yes").sum()
    )
    .reset_index()
)

senior_ethics["Representacion"] = (
    senior_ethics["Clientes"] / len(data) * 100
)

senior_ethics["TasaChurn"] = (
    senior_ethics["Abandonos"]
    / senior_ethics["Clientes"]
    * 100
)

senior_ethics.round(2)

,SeniorCitizen,Clientes,Abandonos,Representacion,TasaChurn
0,0,5901,1393,83.79,23.61
1,1,1142,476,16.21,41.68


La variable `gender` presenta una distribución prácticamente equilibrada en el conjunto de datos, con aproximadamente **49,52 %** de clientes en la categoría `Female` y **50,48 %** en `Male`.

Las tasas de abandono también presentan valores similares entre ambas categorías: aproximadamente **26,92 %** para `Female` y **26,16 %** para `Male`. Por lo tanto, en este análisis descriptivo no se observa una diferencia relevante de `Churn` asociada a `gender`.

En contraste, `SeniorCitizen = 1` representa aproximadamente **16,21 %** de los clientes y presenta una tasa de abandono de **41,68 %**, frente a **23,61 %** en `SeniorCitizen = 0`. Esta diferencia requiere una revisión más profunda antes de realizar cualquier interpretación, especialmente debido al carácter personal de la variable.

### 7.2 Profundización del análisis de `SeniorCitizen`

La variable `SeniorCitizen` presenta diferencias relevantes en la tasa de abandono. Sin embargo, atribuir esta diferencia directamente a la condición de adulto mayor sería una interpretación insuficiente y potencialmente problemática.

Por esta razón, se revisa si ambos grupos presentan también diferencias en características comerciales previamente asociadas con `Churn`, como el tipo de contrato, el servicio de Internet y el método de pago.

In [62]:
# Perfil comparativo de SeniorCitizen según variables previamente asociadas a Churn
senior_profile = pd.DataFrame({
    "Tasa Churn (%)": (
        data.groupby("SeniorCitizen")["Churn"]
        .apply(lambda x: (x == "Yes").mean() * 100)
    ),

    "Contrato mensual (%)": (
        pd.crosstab(
            data["SeniorCitizen"],
            data["Contract"],
            normalize="index"
        )["Month-to-month"] * 100
    ),

    "Fibra óptica (%)": (
        pd.crosstab(
            data["SeniorCitizen"],
            data["InternetService"],
            normalize="index"
        )["Fiber optic"] * 100
    ),

    "Cheque electrónico (%)": (
        pd.crosstab(
            data["SeniorCitizen"],
            data["PaymentMethod"],
            normalize="index"
        )["Electronic check"] * 100
    ),

    "Mediana MonthlyCharges": (
        data.groupby("SeniorCitizen")["MonthlyCharges"].median()
    )
})

senior_profile.round(2)

,Tasa Churn (%),Contrato mensual (%),Fibra óptica (%),Cheque electrónico (%),Mediana MonthlyCharges
SeniorCitizen,,,,,
0,23.61,51.99,38.38,30.01,65.80
1,41.68,70.67,72.77,52.01,84.85


El grupo `SeniorCitizen = 1` presenta una tasa de abandono de aproximadamente **41,68 %**, superior al **23,61 %** observado en `SeniorCitizen = 0`.

Sin embargo, ambos grupos también presentan diferencias importantes en su composición comercial. Entre los clientes clasificados como `SeniorCitizen = 1`, aproximadamente **70,67 %** posee contrato mensual, **72,77 %** utiliza fibra óptica y **52,01 %** utiliza cheque electrónico, categorías que previamente mostraron mayores tasas de abandono.

Por lo tanto, la mayor tasa de `Churn` observada en este grupo no debe atribuirse directamente a la condición representada por `SeniorCitizen`. La variable debe interpretarse con especial precaución para evitar conclusiones simplistas o decisiones potencialmente discriminatorias.

En una eventual etapa de modelamiento, sería necesario evaluar cuidadosamente el impacto del uso de variables personales y comprobar si las decisiones generadas afectan de manera desproporcionada a determinados grupos.

### 7.3 Privacidad, minimización y anonimización

El conjunto de datos no contiene identificadores personales directos como nombre, correo electrónico, teléfono o dirección. Sin embargo, incluye `customerID`, un identificador único asociado a cada cliente.

Esta variable fue utilizada únicamente durante la auditoría de calidad para verificar la unicidad de los registros y posteriormente fue excluida del conjunto de datos preparado, siguiendo un criterio de minimización de datos.

La eliminación de `customerID` reduce el uso de identificadores innecesarios, pero no permite afirmar que el conjunto de datos se encuentre completamente anonimizado. La combinación de distintas características personales y comerciales podría, en determinados contextos y junto con información externa, contribuir a la reidentificación de individuos.

En un entorno real, el tratamiento de los datos debería considerar controles adicionales de acceso, técnicas de anonimización o pseudonimización según el nivel de riesgo y el propósito del análisis.

### 7.4 Consideraciones éticas para un eventual modelamiento

Aunque el entrenamiento de modelos se encuentra fuera del alcance de esta evaluación, el EDA permite identificar aspectos que deberían considerarse antes de una eventual etapa predictiva.

Entre ellos se encuentran:

- evaluar cuidadosamente el uso de variables personales como `gender` y `SeniorCitizen`;
- analizar si determinados grupos se encuentran suficientemente representados;
- comprobar si las predicciones o decisiones presentan diferencias sistemáticas entre grupos;
- evitar interpretar asociaciones exploratorias como relaciones causales;
- utilizar únicamente variables necesarias y pertinentes para el propósito del proyecto;
- mantener medidas adecuadas de privacidad, control de acceso y protección de la información;
- revisar periódicamente si los patrones observados siguen siendo representativos de la población real.

El uso de un eventual modelo de abandono debería orientarse a apoyar acciones beneficiosas para el cliente, como mejorar la experiencia, soporte o condiciones de servicio, y no a justificar prácticas discriminatorias o perjudiciales para segmentos específicos.

### 7.5 Limitaciones del análisis

Los resultados obtenidos corresponden exclusivamente al conjunto de datos analizado y deben interpretarse dentro de dicho contexto.

El dataset no entrega información suficiente para establecer relaciones causales entre las características de los clientes y el abandono. Asimismo, no se dispone en esta etapa de información adicional sobre el contexto temporal de los registros, políticas comerciales, satisfacción del cliente, calidad efectiva del servicio u otros factores externos que podrían influir en `Churn`.

Por esta razón, los hallazgos obtenidos deben considerarse asociaciones exploratorias útiles para orientar investigaciones y decisiones posteriores, y no como evidencia definitiva de causalidad.

## 8. Hallazgos principales del EDA

El análisis exploratorio permitió identificar una serie de patrones relevantes asociados al abandono de clientes. Los siguientes hallazgos sintetizan los resultados de mayor importancia para la comprensión del problema de negocio.

### 8.1 El abandono afecta aproximadamente a 1 de cada 4 clientes

De los **7.043 clientes analizados**, **1.869 presentan `Churn = Yes`**, equivalente a una tasa de abandono de aproximadamente **26,54 %**.

Este resultado establece la magnitud del fenómeno dentro del conjunto de datos y justifica profundizar en las características de los clientes que abandonan.

### 8.2 El mayor abandono se concentra en clientes nuevos con contrato mensual

El análisis conjunto de `Contract`, `tenure` y `Churn` mostró que los clientes con **contrato mensual y hasta 9 meses de permanencia** presentan la mayor tasa de abandono entre los segmentos analizados.

En este grupo, **916 de 1.732 clientes abandonan**, equivalente a aproximadamente **52,89 %**.

Además, dentro de los contratos mensuales la tasa de abandono disminuye a medida que aumenta la permanencia, desde **52,89 %** en los primeros 9 meses hasta aproximadamente **22,05 %** entre clientes con más de 55 meses.

### 8.3 La fibra óptica presenta una elevada tasa de abandono y condiciona la interpretación de los cargos mensuales

Los clientes con **fibra óptica** presentan una tasa de abandono aproximada de **41,89 %**, superior a la tasa global del conjunto de datos.

Además, aproximadamente **69,40 % de los clientes que abandonan utiliza fibra óptica**, servicio que presenta cargos mensuales superiores a DSL y a los clientes sin servicio de Internet.

Este resultado permitió profundizar el hallazgo inicial de `MonthlyCharges`: aunque globalmente quienes abandonan presentan cargos mensuales más elevados, esta diferencia no se mantiene al comparar clientes dentro del mismo tipo de servicio de Internet.

Por lo tanto, los cargos mensuales no deben interpretarse de manera aislada, ya que parte de la diferencia global observada está asociada a la composición de los servicios contratados.

### 8.4 El cheque electrónico presenta la mayor tasa de abandono entre los métodos de pago

Los clientes que utilizan **cheque electrónico** presentan una tasa de abandono aproximada de **45,29 %**, superando en cerca de **18,75 puntos porcentuales** la tasa global del conjunto de datos.

Los demás métodos de pago presentan tasas considerablemente menores: aproximadamente **19,11 %** para cheque por correo, **16,71 %** para transferencia bancaria automática y **15,24 %** para tarjeta de crédito automática.

Este patrón identifica al método de pago como una característica relevante para la segmentación exploratoria, aunque no permite establecer una relación causal con el abandono.

### 8.5 Las diferencias asociadas a `SeniorCitizen` requieren una interpretación contextual y ética

Los clientes clasificados como `SeniorCitizen = 1` presentan una tasa de abandono aproximada de **41,68 %**, frente a **23,61 %** en `SeniorCitizen = 0`.

Sin embargo, este grupo también presenta una mayor concentración de características previamente asociadas con tasas elevadas de abandono, incluyendo contratos mensuales, fibra óptica y cheque electrónico.

Por esta razón, la mayor tasa observada no debe atribuirse directamente a la condición representada por `SeniorCitizen`, ni utilizarse de manera aislada para justificar decisiones que puedan afectar negativamente a este grupo.

### 8.6 La auditoría de calidad permitió identificar una ausencia no detectada inicialmente

La revisión automática mediante `isna()` no identificó valores ausentes. Sin embargo, una inspección adicional de `TotalCharges` permitió detectar **11 registros representados mediante espacios en blanco**, los cuales impedían que la variable fuese almacenada correctamente como numérica.

Los 11 casos presentaban `tenure = 0`, por lo que, después de investigar el patrón, se convirtieron los valores no numéricos a ausentes y posteriormente se asignó `0` exclusivamente a dichos registros.

No se identificaron registros completamente duplicados ni identificadores de clientes repetidos.

## 9. Conclusiones

El análisis exploratorio permitió determinar que el abandono de clientes no se distribuye de manera uniforme dentro del conjunto de datos. La tasa global de `Churn` alcanza aproximadamente **26,54 %**, pero determinados segmentos presentan comportamientos considerablemente diferentes.

La profundización multivariada permitió identificar especialmente a los clientes con **contratos mensuales y baja permanencia** como un segmento de atención relevante, alcanzando una tasa de abandono de aproximadamente **52,89 %** durante los primeros 9 meses.

También se identificaron mayores tasas de abandono entre clientes con fibra óptica y cheque electrónico. Sin embargo, el análisis conjunto demostró la importancia de no interpretar las variables de manera aislada. En particular, la mayor presencia de cargos mensuales entre quienes abandonan se encuentra influenciada por la elevada concentración de clientes con fibra óptica dentro de este grupo.

Los resultados obtenidos corresponden a asociaciones exploratorias y no permiten establecer relaciones causales. Por lo tanto, los hallazgos deben utilizarse como evidencia para orientar investigaciones y decisiones posteriores, evitando conclusiones deterministas sobre el comportamiento individual de los clientes.

Desde la perspectiva de calidad, se identificaron y trataron inconsistencias únicamente cuando existió evidencia que justificara la intervención, manteniendo la trazabilidad de las decisiones realizadas.

Finalmente, el análisis de privacidad y sesgos evidenció la necesidad de utilizar con precaución variables personales como `SeniorCitizen` y de aplicar principios de minimización de datos antes de una eventual continuación del proyecto hacia etapas de modelamiento.

## 10. Metodología CRISP-DM

El proyecto se estructuró utilizando **CRISP-DM (Cross-Industry Standard Process for Data Mining)** como marco metodológico para organizar el proceso de trabajo.

De acuerdo con el alcance definido para esta evaluación, se desarrollaron principalmente las siguientes etapas:

### 10.1 Business Understanding

Se definió el problema de negocio asociado al abandono de clientes, estableciendo como foco del análisis la variable `Churn` y orientando el proyecto hacia la generación de hallazgos útiles para futuras decisiones de retención.

### 10.2 Data Understanding

Se realizó la comprensión inicial del conjunto de datos mediante la revisión de su estructura, tipos de variables y estadísticas descriptivas.

Posteriormente se desarrolló una auditoría de calidad y un análisis exploratorio progresivo, comenzando con análisis univariados y bivariados y profundizando posteriormente mediante cruces multivariados para contextualizar los hallazgos identificados.

### 10.3 Data Preparation

Las decisiones de preparación fueron realizadas únicamente cuando existió evidencia que justificara una modificación.

Se trató la inconsistencia identificada en `TotalCharges`, se validó posteriormente la integridad de los datos y se generó una versión preparada excluyendo `customerID` por corresponder a un identificador único no necesario para posteriores propósitos analíticos.

### Etapas fuera del alcance

Las etapas de **Modeling, Evaluation y Deployment** forman parte de CRISP-DM, pero no se desarrollaron en esta evaluación debido a que el alcance corresponde al análisis exploratorio, preparación y evaluación responsable de los datos.